# Citation Index API: v1.0.0-beta.1 executable end-to-end guide

This notebook is both a tutorial and a final smoke test for a deployment. It runs every supported workflow:

1. upload a PDF to the external **MinerU** extractor,
2. ask **Qwen3.6** to identify the bibliography entries,
3. ask **Qwen3.6** to convert those entries into structured records,
4. link a bounded sample of the parsed citations to external scholarly indexes (rule-based `match` or the GLiNER2 `model` decider),
5. run the **GROBID** path (PDF → raw + parsed references + citing sentences) in one job,
6. extract the metadata of the PDF itself (the citing work) with GROBID and with Qwen, and
7. classify the citing sentences' intent (method / background / result) with the external **CEC** classifier.

Each API operation is asynchronous: submission returns a `job_id`, the client polls `/jobs/{job_id}/status`, and the result is read from `/jobs/{job_id}`. Every route except `/health` requires an `X-API-Key` header when the server sets `API_KEY`. The notebook intentionally fails fast if a service is unavailable or a stage returns an invalid result.

> The combined LLM route `/process/references` is not enabled, so Steps 3–5 compose the LLM stages explicitly. `/process/references/grobid` is the one-call GROBID alternative.

## Goal

A successful top-to-bottom run proves all of the following:

- the API, Redis, storage, and RQ workers are available, and the API key is enforced;
- the workers can reach MinerU, GROBID, and the CEC classifier;
- prompt files are present in the image;
- Qwen returns answer content and valid structured JSON;
- the citation-linking queue returns the requested index IDs and DOIs when found, and reports a failing source per link instead of failing the job;
- parsed references carry the `container_title` field (book or proceedings of a chapter / paper);
- GROBID returns index-aligned raw references, parsed references, and citing contexts;
- both metadata methods return a titled record for the uploaded PDF; and
- job failures, missing job IDs, and invalid requests are surfaced as errors rather than empty successes.

## Setup

Start MinerU in one terminal. This command remains attached while the port-forward is active:

```bash
oc port-forward service/mineru-api 8000:8000
```

Start the Docker stack in a second terminal from the repository root. Compose refuses to start the API unless `API_KEY` is set in `.env`; GROBID and CEC are external services: set `GROBID_ENDPOINT` in `.env` (the compose default `http://grobid:8070` has no matching service) and, if needed, `CEC_ENDPOINT`:

```bash
docker compose up -d --build --scale worker-llm=1 \
  redis api worker-default worker-llm worker-linking
docker compose ps
```

`/link/references?method=model` also needs the linking decider (CPU, ~3.5 GB RAM). Start it by name only when `CITATION_INDEX_LINKING_METHOD=model`:

```bash
docker compose up -d linking-decider
```

The Citation Index API is published on host port **8001** because port **8000** is reserved for MinerU. Docker Compose reads the Qwen endpoint and all keys from `.env`; never put credentials in this notebook.

In [1]:
import json
import os
import time
import uuid
from pathlib import Path

import requests

### Configuration

Defaults target the staging deployment, a small repository fixture, and Matilda linking for at most three extracted references. Override them through environment variables when needed. Set `CITATION_INDEX_LINKING_TARGETS` to `all` or a comma-separated list of `openalex`, `matilda`, `wikidata`, and `opencitations`. `CITATION_INDEX_LINKING_METHOD` is `match` (default, title/author/year rules) or `model` (GLiNER2 decider; needs the `linking-decider` service). `CITATION_INDEX_API_KEY` must match the server's `API_KEY`; leave it empty only against a server with auth disabled.

```bash
export CITATION_INDEX_API=http://localhost:8001
export CITATION_INDEX_API_KEY=...            # the server's API_KEY
export CITATION_INDEX_PDF=/absolute/path/to/paper.pdf
export CITATION_INDEX_LINKING_TARGETS=matilda
export CITATION_INDEX_LINKING_METHOD=match
export CITATION_INDEX_LINKING_LIMIT=3
export CITATION_INDEX_CEC_LIMIT=10
```

In [2]:
API_BASE = os.getenv(
    "CITATION_INDEX_API",
    "https://citation-index-api-graphia-app1-staging.apps.bst2.paas.psnc.pl",
).rstrip("/")
API_KEY = os.getenv("CITATION_INDEX_API_KEY", "")
EXPECTED_VERSION = "1.0.0-beta.1"
PDF_PATH = Path(
    os.getenv("CITATION_INDEX_PDF", "../benchmarks/excite/all_pdfs/27055.pdf")
).expanduser().resolve()
LINKING_TARGETS = os.getenv("CITATION_INDEX_LINKING_TARGETS", "matilda")
LINKING_METHOD = os.getenv("CITATION_INDEX_LINKING_METHOD", "match")
MAX_LINKING_REFERENCES = int(os.getenv("CITATION_INDEX_LINKING_LIMIT", "3"))
MAX_CEC_CITATIONS = int(os.getenv("CITATION_INDEX_CEC_LIMIT", "10"))

POLL_INTERVAL_SECONDS = 2
MAX_JOB_WAIT_SECONDS = 1_800
HTTP_TIMEOUT = (10, 120)  # connect timeout, response timeout

session = requests.Session()
if API_KEY:
    session.headers["X-API-Key"] = API_KEY
print(
    {
        "api": API_BASE,
        "api_key_set": bool(API_KEY),
        "pdf": str(PDF_PATH),
        "pdf_exists": PDF_PATH.is_file(),
        "linking_targets": LINKING_TARGETS,
        "linking_method": LINKING_METHOD,
        "linking_limit": MAX_LINKING_REFERENCES,
        "cec_limit": MAX_CEC_CITATIONS,
    }
)
assert PDF_PATH.is_file(), f"PDF not found: {PDF_PATH}"
assert LINKING_METHOD in {"match", "model"}, "CITATION_INDEX_LINKING_METHOD must be match or model"
assert MAX_LINKING_REFERENCES > 0, "CITATION_INDEX_LINKING_LIMIT must be positive"
assert MAX_CEC_CITATIONS > 0, "CITATION_INDEX_CEC_LIMIT must be positive"

{'api': 'https://citation-index-api-graphia-app1-staging.apps.bst2.paas.psnc.pl', 'api_key_set': True, 'pdf': '/Users/alex/docs/code/Odoma/citation_index/benchmarks/excite/all_pdfs/27055.pdf', 'pdf_exists': True, 'linking_targets': 'matilda', 'linking_method': 'match', 'linking_limit': 3, 'cec_limit': 10}


## Step 1 — Verify service health and authentication

`GET /health` checks the API's Redis and storage dependencies and is the only route that needs no API key (liveness probes carry none). It does not call MinerU, Qwen, GROBID, or CEC, so those integrations are tested by the later stages.

The second check sends a request without the key to a protected route. A server with `API_KEY` set must answer HTTP 401; the check is skipped when no key is configured locally.

In [3]:
health_response = session.get(f"{API_BASE}/health", timeout=HTTP_TIMEOUT)
health_response.raise_for_status()
health = health_response.json()
print(health)
assert health["status"] == "healthy", health
assert health["redis"] == "ok" and health["storage"] == "ok", health
assert health["version"] == EXPECTED_VERSION, health

if API_KEY:
    unauthenticated = requests.get(
        f"{API_BASE}/jobs/{uuid.uuid4()}/status", timeout=HTTP_TIMEOUT
    )
    print({"status_code_without_key": unauthenticated.status_code})
    assert unauthenticated.status_code == 401, unauthenticated.text
    auth_enforced = True
else:
    print("CITATION_INDEX_API_KEY not set: assuming the server runs without auth")
    auth_enforced = None

{'status': 'healthy', 'redis': 'ok', 'storage': 'ok', 'version': '1.0.0-beta.1'}


{'status_code_without_key': 401}


## Step 2 — Understand the asynchronous job contract

Submission endpoints return HTTP 200 with a job object such as:

```json
{"job_id": "...", "status": "queued", "created_at": "..."}
```

A job then moves through `queued` and `processing` to either `completed` or `failed`. The helper below prints only state changes, raises with the server error on failure, and enforces a maximum wait.

In [4]:
def response_json(response: requests.Response) -> dict:
    """Raise an informative HTTP error, then return a JSON object."""
    try:
        payload = response.json()
    except requests.JSONDecodeError as exc:
        preview = response.text[:300].replace("\n", " ")
        raise RuntimeError(
            f"Expected JSON from {response.request.method} {response.url}; "
            f"HTTP {response.status_code}, body={preview!r}"
        ) from exc
    if not response.ok:
        raise RuntimeError(
            f"{response.request.method} {response.url} failed with "
            f"HTTP {response.status_code}: {payload}"
        )
    if not isinstance(payload, dict):
        raise TypeError(f"Expected a JSON object, got {type(payload).__name__}")
    return payload


def wait_for_job(job_id: str, max_wait: int = MAX_JOB_WAIT_SECONDS) -> dict:
    """Poll a job to a terminal state and return its persisted result."""
    started = time.monotonic()
    last_state = None

    while time.monotonic() - started < max_wait:
        status_response = session.get(
            f"{API_BASE}/jobs/{job_id}/status", timeout=HTTP_TIMEOUT
        )
        status_payload = response_json(status_response)
        state = status_payload["status"]

        if state != last_state:
            elapsed = time.monotonic() - started
            stage = status_payload.get("current_stage")
            print(f"[{elapsed:6.1f}s] {job_id}: {state} (stage={stage})")
            last_state = state

        if state == "completed":
            return response_json(
                session.get(f"{API_BASE}/jobs/{job_id}", timeout=HTTP_TIMEOUT)
            )
        if state == "failed":
            raise RuntimeError(
                f"Job {job_id} failed: {status_payload.get('error', 'unknown error')}"
            )

        time.sleep(POLL_INTERVAL_SECONDS)

    raise TimeoutError(f"Job {job_id} did not finish within {max_wait}s")

## Step 3 — Extract Markdown with MinerU

`POST /extract/text` accepts a PDF as multipart form data. Supported extractor names are `pymupdf`, `mineru`, and `grobid`; any other value is rejected with HTTP 422 before a job is queued. This test selects `mineru`, which makes a real call to the configured MinerU service.

> MinerU (`vlm-auto-engine`) can drop footnotes on footnote-style humanities papers; prefer `pymupdf` there.

In [5]:
with PDF_PATH.open("rb") as pdf_file:
    submit_text_response = session.post(
        f"{API_BASE}/extract/text",
        params={"extractor": "mineru", "markdown": "true"},
        files={"file": (PDF_PATH.name, pdf_file, "application/pdf")},
        timeout=HTTP_TIMEOUT,
    )

text_job = response_json(submit_text_response)
print(text_job)
assert text_job["status"] in {"queued", "processing"}

{'job_id': '6acdc638-29da-46e2-9744-118f9571bfeb', 'status': 'queued', 'created_at': '2026-11-25T10:57:19.472059', 'message': 'Text extraction job enqueued'}


In [6]:
text_result = wait_for_job(text_job["job_id"])
extracted_text = text_result.get("text", "")

assert text_result.get("extractor") == "mineru", text_result
assert len(extracted_text) > 1_000, "MinerU returned unexpectedly little text"

print(
    {
        "extractor": text_result.get("extractor"),
        "backend": text_result.get("backend"),
        "mineru_version": text_result.get("mineru_version"),
        "text_characters": len(extracted_text),
    }
)
print("\nPreview:\n", extracted_text[:700].strip())

[   0.0s] 6acdc638-29da-46e2-9744-118f9571bfeb: queued (stage=None)


[   2.1s] 6acdc638-29da-46e2-9744-118f9571bfeb: processing (stage=text_extraction)


[   8.4s] 6acdc638-29da-46e2-9744-118f9571bfeb: completed (stage=text_extraction)
{'extractor': 'mineru', 'backend': None, 'mineru_version': None, 'text_characters': 13031}

Preview:
 # Einleitung zum Schwerpunkt:

# Nationalsozialismus und Krieg in literarischen Autobiographien 30 Jahre „Kindheitsmuster“ von Christa Wolf

Ilse Bürmann, Ortrun Niethammer und Helmut Schmitz

„Kindheitsmuster“, der autobiographische Roman von Christa Wolf, ist im Oktober 1976 im Aufbau-Verlag, Berlin-Ost, erschienen. Nach dem 16. November 1976, d.h. nach der gerade erfolgten Ausbürgerung Wolf Biermanns und dem – maßgeblich von Christa Wolf initiierten – Aufruf der DDR-Schriftsteller, diese Ausbürgerung zurückzunehmen, hätte möglicherweise diese gewichtige und strukturell neue Autobiographie nicht mehr in der DDR erscheinen können. Wolfs Auseinandersetzung mit den Fragen der Teilhabe am Nati


## Step 4 — Extract raw bibliography entries with Qwen

`POST /extract/references` with a JSON `text` body sends the Markdown to the configured medium-intelligence model (`Qwen3.6-27B` by default). `method` is `full_text` (default) or `semantic`. Temperature `0.0` makes this validation run as deterministic as the serving stack permits.

The same route also accepts a multipart PDF with `method=grobid` (see Step 7). Mixing the two — a PDF without `method=grobid`, or `include_parsed` / `include_context` on a JSON body — returns HTTP 400.

In [7]:
submit_extraction_response = session.post(
    f"{API_BASE}/extract/references",
    params={"method": "full_text", "temperature": 0.0},
    json={"text": extracted_text},
    timeout=HTTP_TIMEOUT,
)

extraction_job = response_json(submit_extraction_response)
print(extraction_job)

{'job_id': 'ee6666a4-72b2-4a2a-8eb0-62068ec7e32b', 'status': 'queued', 'created_at': '2026-11-25T10:57:27.975447', 'message': 'Reference extraction job enqueued'}


In [8]:
extraction_result = wait_for_job(extraction_job["job_id"])
raw_references = extraction_result.get("references", [])

assert raw_references, "Qwen returned no references for a document with a bibliography"
assert extraction_result.get("count") == len(raw_references)
assert all(isinstance(item, str) and item.strip() for item in raw_references)

print(f"Extracted {len(raw_references)} bibliography entries:")
for index, reference in enumerate(raw_references[:10], start=1):
    print(f"{index:>2}. {reference[:300]}")

[   0.0s] ee6666a4-72b2-4a2a-8eb0-62068ec7e32b: queued (stage=None)


[   2.1s] ee6666a4-72b2-4a2a-8eb0-62068ec7e32b: processing (stage=reference_extraction)


[  16.6s] ee6666a4-72b2-4a2a-8eb0-62068ec7e32b: completed (stage=reference_extraction)
Extracted 23 bibliography entries:
 1. Assmann, Jan (1997): Das kulturelle Gedächtnis. München
 2. De Bruyn, Günter (1992): Zwischenbilanz. Eine Jugend in Berlin. Frankfurt/M.
 3. Eckstaedt, Anita (1992): Nationalsozialismus in der „zweiten Generation“. Frankfurt/M.
 4. Grass, Günter (2006): Beim Häuten der Zwiebel. Göttingen
 5. Harig, Ludwig (1990): Weh dem, der aus der Reihe tanzt. München
 6. Kuczinski, Rita (1999): Mauerblume. Ein Leben auf der Grenze. München.
 7. Kunert, Günter (2003): Erwachsenenspiele. Erinnerungen. München.
 8. Maron, Monika (1999): Pawels Briefe. Frankfurt/M.
 9. Medicus, Thomas (2005): In den Augen meines Großvaters. München
10. Müller, Heiner (1992/1994): Krieg ohne Schlacht. Leben in zwei Diktaturen. Eine Autobiographie. Köln


## Step 5 — Parse the entries into structured records

`POST /parse/references` accepts a JSON list of strings. With `parser=llm`, Qwen returns fields such as authors, title, year, publication venue, volume, issue, and identifiers when they are present in the source citation. `container_title` holds the book that contains a chapter or the proceedings that contain a conference paper; `journal_title` keeps journals and other serials. `parser=grobid` is the alternative; other values return HTTP 422. A custom `prompt_name` must name an existing file under `prompts/` (HTTP 400 otherwise).

In [9]:
submit_parse_response = session.post(
    f"{API_BASE}/parse/references",
    params={"parser": "llm", "temperature": 0.0},
    json={"references": raw_references},
    timeout=HTTP_TIMEOUT,
)

parse_job = response_json(submit_parse_response)
print(parse_job)

{'job_id': 'da9f1526-6733-4e93-9695-50254d285764', 'status': 'queued', 'created_at': '2026-11-25T10:57:44.693807', 'message': 'Reference parsing job enqueued'}


In [10]:
parse_result = wait_for_job(parse_job["job_id"])
structured_references = parse_result.get("references", [])

assert parse_result.get("parser") == "llm", parse_result
assert parse_result.get("count") == len(structured_references)
assert len(structured_references) == len(raw_references)
assert all(isinstance(item, dict) for item in structured_references)
assert all("container_title" in item for item in structured_references), "container_title missing"

# Counts alone cannot detect a field-level regression: one record per input string is
# returned even when every title is null. Require most references to carry a title.
MIN_TITLED_SHARE = 0.8

untitled = [
    reference
    for reference in structured_references
    if not (reference.get("full_title") or reference.get("journal_title"))
]
titled_share = 1 - len(untitled) / len(structured_references)
assert titled_share >= MIN_TITLED_SHARE, (
    f"only {titled_share:.0%} of references carry a title, expected at least "
    f"{MIN_TITLED_SHARE:.0%}; untitled entries: {untitled}"
)

print(f"Parsed {len(structured_references)} structured references, {titled_share:.0%} titled.")
print(json.dumps(structured_references[:2], indent=2, ensure_ascii=False)[:4_000])

[   0.0s] da9f1526-6733-4e93-9695-50254d285764: queued (stage=None)


[   2.1s] da9f1526-6733-4e93-9695-50254d285764: processing (stage=reference_parsing)


[ 118.5s] da9f1526-6733-4e93-9695-50254d285764: completed (stage=reference_parsing)
Parsed 23 structured references, 100% titled.
[
  {
    "full_title": "Das kulturelle Gedächtnis",
    "container_title": null,
    "journal_title": null,
    "authors": [
      {
        "first_name": "Jan",
        "middle_name": null,
        "surname": "Assmann",
        "name_link": null,
        "role_name": null
      }
    ],
    "editors": null,
    "publisher": null,
    "translator": null,
    "publication_place": "München",
    "publication_year": 1997,
    "publication_date_raw": "1997",
    "identifiers": [],
    "ref_type": "book",
    "raw": {},
    "volume": null,
    "issue": null,
    "pages": null,
    "cited_range": null,
    "footnote_number": null
  },
  {
    "full_title": "Zwischenbilanz. Eine Jugend in Berlin",
    "container_title": null,
    "journal_title": null,
    "authors": [
      {
        "first_name": "Günter",
        "middle_name": null,
        "surname": "De Bruy

## Step 6 — Link citations to scholarly indexes

`POST /link/references` accepts a parsed reference object with required `full_title`, non-empty `authors`, and integer `publication_year` fields. With `batched=true`, `reference` must be a non-empty JSON array of these objects. Raw strings and objects missing a field are rejected with HTTP 422; an array without `batched=true` (or the reverse) and an unknown target are rejected with HTTP 400. Candidates are searched using parsed fields and filtered with `custom_match()`: title similarity ≥90 and either first-author similarity ≥70 or year within ±1.

`method=match` (default) applies those rules. `method=model` lets the fine-tuned GLiNER2 decider choose among each index's search results, or none; it needs the original citation string as `raw_reference` on every reference (HTTP 400 otherwise), and each link gains a `score`. A decider failure fails the whole job. `filter_reviews_by_doi=true` drops candidates whose DOI only names a book review (Choice Reviews Online, `10.5860/choice.*`) before either method chooses.

The `target` query parameter accepts `openalex`, `matilda`, `wikidata`, `opencitations`, `all`, or a comma-separated list. OpenCitations returns OMIDs (`br/...`) and is the slowest target (SPARQL fallback, several seconds per reference). One failing source does not fail the job: its link comes back as `{"id": null, "doi": null, "error": "..."}`. The job fails only when every lookup fails.

This executable check takes the structured records from Step 5, selects records with the required title, first-author name, and year, and submits those three fields plus the Step 4 string as `raw_reference` (Steps 4 and 5 are index-aligned). It links at most `MAX_LINKING_REFERENCES` eligible citations to keep live external requests bounded. A target can legitimately return `null` for both fields when it cannot make a confident match. The default target is Matilda; it needs the deployment's Matilda credentials on the linking worker. OpenAlex is not the default because shared OpenAlex quotas can return HTTP 429, which fails a single-target job.

In [11]:
eligible_references = []
for raw_reference, reference in zip(raw_references, structured_references):
    title = reference.get("full_title")
    authors = reference.get("authors") or []
    year = reference.get("publication_year")
    first_author = authors[0] if authors else None
    author_name = (
        first_author
        if isinstance(first_author, str)
        else (first_author.get("surname") or first_author.get("name"))
        if isinstance(first_author, dict)
        else None
    )
    if (
        isinstance(title, str) and title.strip()
        and isinstance(author_name, str) and author_name.strip()
        and type(year) is int and 1 <= year <= 9999
    ):
        eligible_references.append({
            "full_title": title,
            "authors": [
                {key: value for key, value in author.items() if value is not None}
                if isinstance(author, dict) else author
                for author in authors
            ],
            "publication_year": year,
            "raw_reference": raw_reference,
        })

print(f"Skipped {len(structured_references) - len(eligible_references)} records missing linking fields")
references_to_link = eligible_references[:MAX_LINKING_REFERENCES]
assert references_to_link, "No parsed references have the required title, author, and year"

submit_link_response = session.post(
    f"{API_BASE}/link/references",
    params={"batched": "true", "target": LINKING_TARGETS, "method": LINKING_METHOD},
    json={"reference": references_to_link},
    timeout=HTTP_TIMEOUT,
)

link_job = response_json(submit_link_response)
print(link_job)

Skipped 0 records missing linking fields
{'job_id': '636b01a2-53b0-4556-ae63-11dadf7ebed7', 'status': 'queued', 'created_at': '2026-11-25T10:59:43.293822', 'message': 'Citation linking job enqueued'}


In [12]:
link_result = wait_for_job(link_job["job_id"])
linked_references = link_result.get("results", [])
requested_targets = [
    target.strip().lower()
    for target in LINKING_TARGETS.split(",")
    if target.strip()
]
expected_linking_targets = (
    ["openalex", "matilda", "wikidata", "opencitations"]
    if "all" in requested_targets
    else list(dict.fromkeys(requested_targets))
)

assert link_result.get("count") == len(references_to_link), link_result
assert link_result.get("method") == LINKING_METHOD, link_result
assert link_result.get("filter_reviews_by_doi") is False, link_result
assert link_result.get("targets") == expected_linking_targets, link_result
assert len(linked_references) == len(references_to_link), link_result
assert [item.get("reference") for item in linked_references] == references_to_link
for item in linked_references:
    assert list(item.get("links", {})) == expected_linking_targets, item
    for link in item["links"].values():
        assert {"id", "doi"} <= set(link) <= {"id", "doi", "error", "score"}, link
        if LINKING_METHOD == "model" and "error" not in link and link["id"]:
            assert isinstance(link.get("score"), (int, float)), link
        assert link["id"] is None or isinstance(link["id"], str), link
        assert link["doi"] is None or isinstance(link["doi"], str), link

linked_id_count = sum(
    link["id"] is not None
    for item in linked_references
    for link in item["links"].values()
)
source_errors = [
    (target, link["error"])
    for item in linked_references
    for target, link in item["links"].items()
    if link.get("error")
]
if source_errors:
    print(f"WARNING: {len(source_errors)} per-source linking errors: {source_errors}")
print(
    json.dumps(
        {"linked_id_count": linked_id_count, "results": linked_references},
        indent=2,
        ensure_ascii=False,
    )
)

[   0.0s] 636b01a2-53b0-4556-ae63-11dadf7ebed7: queued (stage=None)


[ 406.7s] 636b01a2-53b0-4556-ae63-11dadf7ebed7: processing (stage=citation_linking)


[ 408.9s] 636b01a2-53b0-4556-ae63-11dadf7ebed7: completed (stage=citation_linking)
{
  "linked_id_count": 1,
  "results": [
    {
      "reference": {
        "full_title": "Das kulturelle Gedächtnis",
        "authors": [
          {
            "first_name": "Jan",
            "surname": "Assmann"
          }
        ],
        "publication_year": 1997,
        "raw_reference": "Assmann, Jan (1997): Das kulturelle Gedächtnis. München"
      },
      "links": {
        "matilda": {
          "id": "709525a3-42ab-4e19-b376-3546dbd71f49",
          "doi": "10.17104/9783406703409"
        }
      }
    },
    {
      "reference": {
        "full_title": "Zwischenbilanz. Eine Jugend in Berlin",
        "authors": [
          {
            "first_name": "Günter",
            "surname": "De Bruyn"
          }
        ],
        "publication_year": 1992,
        "raw_reference": "De Bruyn, Günter (1992): Zwischenbilanz. Eine Jugend in Berlin. Frankfurt/M."
      },
      "links": {
        "

## Step 7 — One-call GROBID path with citing contexts

`POST /process/references/grobid` uploads the PDF to GROBID fulltext and returns, in one job on the `default` queue:

- `references` — raw reference strings,
- `parsed_references` — structured records (or `null` when GROBID could not parse an entry), with `ref_type` (`journal-article`, `book-chapter`, `book`, `proceedings-paper`, or none) and `container_title` read from the TEI structure, and
- with `include_context=true`, `contexts` — per reference, the sentences that cite it as `{section, citation, marker, ref_id}`.

The three lists are index-aligned. The same result is available from `POST /extract/references?method=grobid&include_parsed=true&include_context=true` with a multipart PDF. No LLM is involved, so this path is much faster than Steps 3–5, but it depends on GROBID's segmentation. GROBID responses are cached per PDF hash, so later GROBID calls on the same PDF (Step 8) skip GROBID.

In [13]:
with PDF_PATH.open("rb") as pdf_file:
    submit_grobid_response = session.post(
        f"{API_BASE}/process/references/grobid",
        params={"include_context": "true"},
        files={"file": (PDF_PATH.name, pdf_file, "application/pdf")},
        timeout=HTTP_TIMEOUT,
    )

grobid_job = response_json(submit_grobid_response)
print(grobid_job)

{'job_id': '646f6118-31ba-4569-b34c-da2942e560ef', 'status': 'queued', 'created_at': '2026-11-25T11:06:32.358007', 'message': 'GROBID reference extraction job enqueued'}


In [14]:
grobid_result = wait_for_job(grobid_job["job_id"])
grobid_references = grobid_result.get("references", [])
grobid_parsed = grobid_result.get("parsed_references", [])
grobid_contexts = grobid_result.get("contexts", [])

assert grobid_result.get("method") == "grobid", grobid_result
assert grobid_references, "GROBID returned no references"
assert grobid_result.get("count") == len(grobid_references)
assert len(grobid_parsed) == len(grobid_references) == len(grobid_contexts)
assert all(item is None or isinstance(item, dict) for item in grobid_parsed)
for reference_contexts in grobid_contexts:
    for context in reference_contexts:
        assert {"section", "citation", "marker", "ref_id"} <= set(context), context

cited_count = sum(bool(reference_contexts) for reference_contexts in grobid_contexts)
print(
    f"GROBID: {len(grobid_references)} references, "
    f"{sum(item is not None for item in grobid_parsed)} parsed, "
    f"{cited_count} with at least one citing sentence"
)
for raw, parsed, reference_contexts in list(
    zip(grobid_references, grobid_parsed, grobid_contexts)
)[:3]:
    print("\n-", raw[:200])
    parsed = parsed or {}
    print("  title:", parsed.get("full_title"), "| type:", parsed.get("ref_type"),
          "| container:", parsed.get("container_title"), "| pages:", parsed.get("pages"))
    for context in reference_contexts[:2]:
        print(f"  cited in [{context['section'] or '?'}]: {context['citation'][:200]}")

[   0.0s] 646f6118-31ba-4569-b34c-da2942e560ef: queued (stage=None)


[   2.1s] 646f6118-31ba-4569-b34c-da2942e560ef: completed (stage=reference_extraction)
GROBID: 4 references, 4 parsed, 2 with at least one citing sentence

- Assmann, Jan (1997): Das kulturelle Gedächtnis. München De Bruyn, Günter (1992): Zwischenbilanz. Eine Jugend in Berlin. Frankfurt/M. Eckstaedt, Anita (1992): Nationalsozialismus in der ‚zweiten Genera
  title: Nationalsozialismus in der ‚zweiten Generation | type: book-chapter | container: Weh dem, der aus der Reihe tanzt | pages: None

- Kunert, Günter (2003): Erwachsenenspiele. Erinnerungen. München.
  title: Erwachsenenspiele. Erinnerungen | type: book | container: None | pages: None

- Maron, Monika (1999): Pawels Briefe. Frankfurt/M. Medicus, Thomas (2005): In den Augen meines Großvaters. München Müller, Heiner (1992/1994): Krieg ohne Schlacht. Leben in zwei Diktaturen. Eine Autobi
  title: Krieg ohne Schlacht. Leben in zwei Diktaturen. Eine Autobiographie | type: book-chapter | container: Die deutsche Autobiographie im 18. J

## Step 8 — Extract the metadata of the PDF itself

`POST /extract/metadata` returns the bibliographic record of the uploaded document (the citing work), as `{"metadata": {...}, "method": "..."}` where `metadata` has the same fields as a parsed reference. `method=grobid` (default) reads the TEI header of GROBID's fulltext on the `default` queue; it reuses the cached TEI from Step 7. `method=llm` sends the first two pages to Qwen on the `llm-tasks` queue. Only PDF uploads are accepted (HTTP 400 otherwise).

Both methods run here; each must return a record with a non-empty title.


In [15]:
METADATA_METHODS = ("grobid", "llm")

metadata_jobs = {}
for method in METADATA_METHODS:
    with PDF_PATH.open("rb") as pdf_file:
        metadata_jobs[method] = response_json(
            session.post(
                f"{API_BASE}/extract/metadata",
                params={"method": method},
                files={"file": (PDF_PATH.name, pdf_file, "application/pdf")},
                timeout=HTTP_TIMEOUT,
            )
        )
    print(metadata_jobs[method])

metadata_results = {}
for method, job in metadata_jobs.items():
    result = wait_for_job(job["job_id"])
    assert result.get("method") == method, result
    metadata = result.get("metadata")
    assert isinstance(metadata, dict), result
    assert (metadata.get("full_title") or "").strip(), f"{method} metadata has no title: {metadata}"
    metadata_results[method] = metadata

for method, metadata in metadata_results.items():
    first_author = (metadata.get("authors") or [None])[0]
    print(
        f"{method:>6}: {metadata.get('full_title')!r} | year={metadata.get('publication_year')} "
        f"| first author={first_author} | publisher={metadata.get('publisher')}"
    )


{'job_id': '91281f39-5ad2-4d42-ba7b-d72cf1655e49', 'status': 'queued', 'created_at': '2026-11-25T11:06:34.568682', 'message': 'Metadata extraction job enqueued'}
{'job_id': '880c5f3e-d855-410a-ae0c-e1f056b655de', 'status': 'queued', 'created_at': '2026-11-25T11:06:34.610327', 'message': 'Metadata extraction job enqueued'}
[   0.0s] 91281f39-5ad2-4d42-ba7b-d72cf1655e49: completed (stage=metadata_extraction)


[   0.0s] 880c5f3e-d855-410a-ae0c-e1f056b655de: processing (stage=metadata_extraction)


[  10.4s] 880c5f3e-d855-410a-ae0c-e1f056b655de: completed (stage=metadata_extraction)
grobid: 'Einleitung zum Schwerpunkt' | year=None | first author={'first_name': 'Christa', 'middle_name': None, 'surname': 'Wolf', 'name_link': None, 'role_name': None} | publisher=None
   llm: 'Einleitung zum Schwerpunkt: Nationalsozialismus und Krieg in literarischen Autobiographien 30 Jahre „Kindheitsmuster“ von Christa Wolf' | year=2006 | first author={'first_name': 'Ilse', 'middle_name': None, 'surname': 'Bürmann', 'name_link': None, 'role_name': None} | publisher=Verlag Barbara Budrich


## Step 9 — Classify citation intent with CEC

`POST /classify/citations` takes `{"citations": [{"section": "...", "citation": "..."}, ...]}` and returns one result per sentence, in input order. `mode` is `WS` (with sections), `WoS` (without), or `M` (default: per item, with section when non-empty). GROBID contexts from Step 7 can be posted as-is; extra keys such as `marker` and `ref_id` are ignored.

CEC serves one request at a time and rebuilds its predictor per call, so send a paper's citations in **one** request rather than one per sentence. This check sends at most `MAX_CEC_CITATIONS` sentences. If GROBID resolved no citing sentences in the PDF (common for short briefs without inline markers), one fixed example sentence is sent instead so CEC is always exercised; `cec_source` in the verdict says which input was used.

In [16]:
citations_to_classify = [
    {"section": context["section"], "citation": context["citation"]}
    for reference_contexts in grobid_contexts
    for context in reference_contexts
][:MAX_CEC_CITATIONS]
cec_source = "grobid"
if not citations_to_classify:
    print("No citing sentences found by GROBID; classifying a fixed example sentence")
    citations_to_classify = [
        {"section": "Methods", "citation": "We follow the approach of Smith et al. (2020)."}
    ]
    cec_source = "fallback"

submit_cec_response = session.post(
    f"{API_BASE}/classify/citations",
    params={"mode": "M"},
    json={"citations": citations_to_classify},
    timeout=HTTP_TIMEOUT,
)
cec_job = response_json(submit_cec_response)
print(cec_job)

cec_result = wait_for_job(cec_job["job_id"])
cec_labels = cec_result.get("results", [])
assert cec_result.get("mode") == "M", cec_result
assert cec_result.get("count") == len(citations_to_classify), cec_result
assert len(cec_labels) == len(citations_to_classify), cec_result

for citation, label in zip(citations_to_classify, cec_labels):
    print(f"{json.dumps(label, ensure_ascii=False)[:120]}  <-  {citation['citation'][:120]}")

{'job_id': '6f3ba11d-afce-4c6a-808d-a2f8963f60fd', 'status': 'queued', 'created_at': '2026-11-25T11:06:45.216562', 'message': 'Citation classification job enqueued'}


[   0.0s] 6f3ba11d-afce-4c6a-808d-a2f8963f60fd: queued (stage=None)


[   2.1s] 6f3ba11d-afce-4c6a-808d-a2f8963f60fd: processing (stage=citation_classification)


[  33.2s] 6f3ba11d-afce-4c6a-808d-a2f8963f60fd: completed (stage=citation_classification)
{"SECTION": "", "CITATION": "Maron 1999, Kuczynski 1999, Timm 2003) bzw.", "SCIBERT MET POSITIVE PROBABILITY": 0.0263458  <-  Maron 1999, Kuczynski 1999, Timm 2003) bzw.
{"SECTION": "", "CITATION": "Erst seit den 1990er Jahren werden -sowohl wissenschaftlich als auch literarisch -die traum  <-  Erst seit den 1990er Jahren werden -sowohl wissenschaftlich als auch literarisch -die traumatischen Langzeitfolgen von N
{"SECTION": "", "CITATION": "Auch westdeutsche Angehörige dieser Generation beziehen sich in ihren Erinnerungen implizit  <-  Auch westdeutsche Angehörige dieser Generation beziehen sich in ihren Erinnerungen implizit entweder nachfolgend (Harig 


## Checks — verify the error contract

A missing job must return HTTP 404. In production code, use `response_json` or equivalent handling so failed jobs and invalid IDs cannot be mistaken for empty results.

Invalid requests must be rejected before a job is created:

- linking: raw strings and parsed objects missing a required field → 422; an array without `batched=true`, an unknown target, or `method=model` without `raw_reference` → 400;
- unknown `extractor` / `parser` / `method` values → 422;
- `include_context` on a JSON (LLM) extraction request → 400;
- an empty `citations` list for CEC, or an unknown `mode` → 422 / 400;
- a non-PDF upload to `/extract/metadata` → 400.

In [17]:
missing_job_id = str(uuid.uuid4())
missing_response = session.get(
    f"{API_BASE}/jobs/{missing_job_id}/status", timeout=HTTP_TIMEOUT
)
print({"status_code": missing_response.status_code, "body": missing_response.json()})
assert missing_response.status_code == 404

invalid_linking_inputs = ["Smith. A paper. 2024."]
for field in ("full_title", "authors", "publication_year"):
    incomplete_reference = dict(references_to_link[0])
    del incomplete_reference[field]
    invalid_linking_inputs.append(incomplete_reference)

for invalid_reference in invalid_linking_inputs:
    invalid_response = session.post(
        f"{API_BASE}/link/references",
        json={"reference": invalid_reference},
        timeout=HTTP_TIMEOUT,
    )
    assert invalid_response.status_code == 422, invalid_response.text
print("Invalid linking inputs rejected with HTTP 422")

# (method, path, params, json body, expected status)
invalid_requests = [
    ("link: array without batched", "/link/references", {}, {"reference": references_to_link[:1]}, 400),
    ("link: unknown target", "/link/references", {"target": "crossref"}, {"reference": references_to_link[0]}, 400),
    ("link: model without raw_reference", "/link/references", {"method": "model"}, {"reference": {key: value for key, value in references_to_link[0].items() if key != "raw_reference"}}, 400),
    ("extract: unknown method", "/extract/references", {"method": "page_by_page"}, {"text": "x"}, 422),
    ("extract: context without grobid", "/extract/references", {"include_context": "true"}, {"text": "x"}, 400),
    ("parse: unknown parser", "/parse/references", {"parser": "regex"}, {"references": ["x"]}, 422),
    ("classify: empty citations", "/classify/citations", {}, {"citations": []}, 422),
    ("classify: unknown mode", "/classify/citations", {"mode": "XX"}, {"citations": [{"citation": "x"}]}, 400),
]
for label, path, params, body, expected in invalid_requests:
    response = session.post(f"{API_BASE}{path}", params=params, json=body, timeout=HTTP_TIMEOUT)
    assert response.status_code == expected, (label, response.status_code, response.text)
print(f"{len(invalid_requests)} invalid requests rejected with the expected 400/422")

non_pdf_response = session.post(
    f"{API_BASE}/extract/metadata",
    files={"file": ("notes.txt", b"not a pdf", "text/plain")},
    timeout=HTTP_TIMEOUT,
)
assert non_pdf_response.status_code == 400, non_pdf_response.text
print("Non-PDF metadata upload rejected with HTTP 400")

{'status_code': 404, 'body': {'detail': 'Job c644da8c-887a-4c10-8465-2161f77e8c5b not found'}}


Invalid linking inputs rejected with HTTP 422


8 invalid requests rejected with the expected 400/422
Non-PDF metadata upload rejected with HTTP 400


## Checks — final end-to-end verdict

This cell deliberately contains assertions, not just display logic. If it prints `PASS`, the tested PDF completed every supported stage and all required outputs were persisted.

In [18]:
end_to_end_checks = {
    "api_version": health["version"] == EXPECTED_VERSION,
    "api_healthy": health["status"] == "healthy",
    "mineru_text_returned": len(extracted_text) > 1_000,
    "raw_references_returned": len(raw_references) > 0,
    "all_references_parsed": len(structured_references) == len(raw_references),
    "references_carry_titles": titled_share >= MIN_TITLED_SHARE,
    "all_linking_results_returned": len(linked_references) == len(references_to_link),
    "linking_shape_is_valid": all(
        list(item.get("links", {})) == expected_linking_targets
        for item in linked_references
    ),
    "grobid_references_returned": len(grobid_references) > 0,
    "metadata_titled": all(
        (metadata_results[method].get("full_title") or "").strip() for method in METADATA_METHODS
    ),
    "grobid_outputs_aligned": len(grobid_parsed) == len(grobid_references) == len(grobid_contexts),
    "cec_results_aligned": len(cec_labels) == len(citations_to_classify),
    "missing_job_returns_404": missing_response.status_code == 404,
}
assert all(end_to_end_checks.values()), end_to_end_checks

verdict = {
    "verdict": "PASS",
    "version": health["version"],
    "auth_enforced": auth_enforced,
    "pdf": PDF_PATH.name,
    "text_characters": len(extracted_text),
    "raw_reference_count": len(raw_references),
    "structured_reference_count": len(structured_references),
    "titled_share": round(titled_share, 3),
    "linking_reference_count": len(linked_references),
    "linking_targets": expected_linking_targets,
    "linking_method": LINKING_METHOD,
    "linked_id_count": linked_id_count,
    "linking_source_errors": len(source_errors),
    "grobid_reference_count": len(grobid_references),
    "grobid_cited_reference_count": cited_count,
    "metadata_titles": {method: metadata_results[method].get("full_title") for method in METADATA_METHODS},
    "cec_classified_count": len(cec_labels),
    "cec_source": cec_source,
    "jobs": {
        "text_extraction": text_job["job_id"],
        "reference_extraction": extraction_job["job_id"],
        "reference_parsing": parse_job["job_id"],
        "citation_linking": link_job["job_id"],
        "grobid_reference_pipeline": grobid_job["job_id"],
        **{f"metadata_{method}": job["job_id"] for method, job in metadata_jobs.items()},
        "citation_classification": cec_job["job_id"],
    },
}
print(json.dumps(verdict, indent=2))

{
  "verdict": "PASS",
  "version": "1.0.0-beta.1",
  "auth_enforced": true,
  "pdf": "27055.pdf",
  "text_characters": 13031,
  "raw_reference_count": 23,
  "structured_reference_count": 23,
  "titled_share": 1.0,
  "linking_reference_count": 3,
  "linking_targets": [
    "matilda"
  ],
  "linking_method": "match",
  "linked_id_count": 1,
  "linking_source_errors": 0,
  "grobid_reference_count": 4,
  "grobid_cited_reference_count": 2,
  "metadata_titles": {
    "grobid": "Einleitung zum Schwerpunkt",
    "llm": "Einleitung zum Schwerpunkt: Nationalsozialismus und Krieg in literarischen Autobiographien 30 Jahre \u201eKindheitsmuster\u201c von Christa Wolf"
  },
  "cec_classified_count": 3,
  "cec_source": "grobid",
  "jobs": {
    "text_extraction": "6acdc638-29da-46e2-9744-118f9571bfeb",
    "reference_extraction": "ee6666a4-72b2-4a2a-8eb0-62068ec7e32b",
    "reference_parsing": "da9f1526-6733-4e93-9695-50254d285764",
    "citation_linking": "636b01a2-53b0-4556-ae63-11dadf7ebed7",
   

## Equivalent command-line workflow

Every request except `/health` carries the API key:

```bash
export CITATION_INDEX_API_KEY=...
```

Submit the first stage:

```bash
curl -H "X-API-Key: $CITATION_INDEX_API_KEY" \
  -F "file=@/absolute/path/paper.pdf;type=application/pdf" \
  "http://localhost:8001/extract/text?extractor=mineru&markdown=true"
```

Poll and retrieve any returned job ID:

```bash
curl -H "X-API-Key: $CITATION_INDEX_API_KEY" http://localhost:8001/jobs/JOB_ID/status
curl -H "X-API-Key: $CITATION_INDEX_API_KEY" http://localhost:8001/jobs/JOB_ID
```

Submit a parsed citation batch to more than one linking target:

```bash
curl -X POST -H "X-API-Key: $CITATION_INDEX_API_KEY" \
  "http://localhost:8001/link/references?batched=true&target=matilda,opencitations" \
  -H "Content-Type: application/json" \
  -d '{"reference":[{"full_title":"Deep learning","authors":["LeCun, Y."],"publication_year":2015}]}'
```

The same batch through the GLiNER2 decider, which needs the original citation string:

```bash
curl -X POST -H "X-API-Key: $CITATION_INDEX_API_KEY" \
  "http://localhost:8001/link/references?batched=true&target=openalex&method=model" \
  -H "Content-Type: application/json" \
  -d '{"reference":[{"full_title":"Deep learning","authors":["LeCun, Y."],"publication_year":2015,"raw_reference":"LeCun, Y., Bengio, Y., Hinton, G. (2015). Deep learning. Nature 521, 436-444."}]}'
```

Run the GROBID path with citing contexts, read the PDF's own metadata, then classify sentences:

```bash
curl -H "X-API-Key: $CITATION_INDEX_API_KEY" \
  -F "file=@/absolute/path/paper.pdf;type=application/pdf" \
  "http://localhost:8001/process/references/grobid?include_context=true"

curl -H "X-API-Key: $CITATION_INDEX_API_KEY" \
  -F "file=@/absolute/path/paper.pdf;type=application/pdf" \
  "http://localhost:8001/extract/metadata?method=grobid"

curl -X POST -H "X-API-Key: $CITATION_INDEX_API_KEY" \
  "http://localhost:8001/classify/citations?mode=M" \
  -H "Content-Type: application/json" \
  -d '{"citations":[{"section":"Methods","citation":"We follow the approach of Smith et al. (2020)."}]}'
```

For the other JSON stages, write the previous response to a file or use `jq` to build the next request. Python is usually clearer for the complete chain because extracted document text can be large.

## API quick reference

All routes except `/health` require `X-API-Key` when the server sets `API_KEY`.

| Method | Path | Queue | Input | Result |
|---|---|---|---|---|
| `GET` | `/health` | — | — | service health (no key needed) |
| `POST` | `/extract/text` | `default` | multipart PDF; `extractor=pymupdf\|mineru\|grobid` | Markdown text |
| `POST` | `/extract/references` | `llm-tasks` | `{"text": "..."}`; `method=full_text\|semantic` | raw citation strings |
| `POST` | `/extract/references` | `default` | multipart PDF; `method=grobid`, `include_parsed`, `include_context` | raw strings (+ parsed, + contexts) |
| `POST` | `/parse/references` | `llm-tasks` or `default` | `{"references": [...]}`; `parser=llm\|grobid` | structured records |
| `POST` | `/link/references` | `linking` | `{"reference": {"full_title": "...", "authors": ["Smith"], "publication_year": 2024}}` or object array with `batched=true`; `target=openalex\|matilda\|wikidata\|opencitations\|all`; `method=match\|model` (`model` needs `raw_reference`); `filter_reviews_by_doi` | target IDs and DOIs (per-source `error` on failure; `score` with `model`) |
| `POST` | `/process/references/grobid` | `default` | multipart PDF; `include_context` | raw + parsed references (+ contexts) |
| `POST` | `/extract/metadata` | `default` (`grobid`) or `llm-tasks` (`llm`) | multipart PDF; `method=grobid\|llm` | metadata of the PDF itself |
| `POST` | `/classify/citations` | `default` | `{"citations": [{"section": "...", "citation": "..."}]}`; `mode=WS\|WoS\|M` | one intent label per sentence |
| `GET` | `/jobs/{job_id}/status` | — | job ID | status and error metadata |
| `GET` | `/jobs/{job_id}` | — | completed job ID | persisted result |

Interactive OpenAPI documentation is available at [`http://localhost:8001/docs`](http://localhost:8001/docs); the generated spec is `api_1.0.0-beta.1.yml`.